In [1]:
# Parameters
BATCH_MODE = True


# Doctor vs ChatGPT: Multi-Agent Medical Chatbot

**Navigation**: [Index](../README.md)

**Estimated duration**: 30 minutes | **Prerequisites**: OpenAI account with API key, basic knowledge of Python

In this use case, we build a simulated medical consultation system where three AI agents cooperate: a **general practitioner** who asks questions, a **medical AI** that analyzes symptoms, and a **pharmacist** who recommends treatments. The dialogue is orchestrated by Semantic Kernel via `AgentGroupChat`, with an automatic termination strategy.

> **Warning**: this notebook is an educational exercise. Diagnoses and medication recommendations are simulated and in no way replace a real medical consultation.

In [2]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')


## Importing libraries

This use case implements a **multi-agent conversation** in the medical field. Three agents cooperate: a general practitioner, an AI specialized in diagnosis, and a pharmacist. Each agent has its own system prompt and dedicated plugins via `@kernel_function`.

**Educational objectives**:
- Understand multi-agent orchestration with `AgentGroupChat`
- Use `@kernel_function` plugins to equip agents with specific capabilities
- Implement a `TerminationStrategy` to control the end of the dialogue
- Configure `FunctionChoiceBehavior.Auto()` for automatic plugin calls

In [3]:
import os
import logging
import asyncio
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies import TerminationStrategy
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import ChatHistory
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.connectors.ai import FunctionChoiceBehavior
from typing import Annotated
print("Imports OK")
print("Imports et configuration OK")


Imports OK
Imports et configuration OK


In [4]:
# Charger les variables d'environnement
load_dotenv()

True

## Log configuration

The `logging` module makes it possible to trace exchanges between agents in real time. Each message will be timestamped and identified by the name of the sending agent, which makes debugging multi-agent conversations easier.

In [5]:
# Configuration des logs
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger('MedicalAI')
print("Configuration des logs OK")


Configuration des logs OK


## Creating the Semantic Kernel kernel

In [6]:
# Création du kernel Semantic Kernel
def create_kernel():
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(
        service_id="openai",
        ai_model_id="gpt-4o-mini",  # Modifier si besoin
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel
print("Kernel Semantic Kernel créé")
print("Imports OK")
print("Fonction create_kernel definie")


Kernel Semantic Kernel créé
Imports OK
Fonction create_kernel definie


## Medical plugins: equipping agents with specific skills

Each agent has a dedicated plugin containing functions annotated with `@kernel_function`. These functions act as tools that the LLM can automatically invoke when it needs them:

- **DoctorPlugin**: asks follow-up questions based on the mentioned symptom
- **MedicalAIPlugin**: assesses the severity of a symptom (Mild, Moderate, Severe, Critical)
- **PharmacistPlugin**: recommends an appropriate medication with dosage and precautions

The kernel is the central element of Semantic Kernel. The `create_kernel()` function instantiates a kernel and registers an `OpenAIChatCompletion` service in it, which will provide access to the GPT-4o-mini model. Each agent will use this shared kernel to generate its responses.

In [7]:
class DoctorPlugin:
    """Plugin permettant au médecin de poser des questions complémentaires sur les symptômes."""
    @kernel_function(description="Pose des questions supplémentaires pour affiner le diagnostic.")
    def ask_followup_questions(self, symptom: Annotated[str, "Symptôme décrit par l'utilisateur"]) -> str:
        """Retourne une question en fonction du symptôme mentionné."""
        questions_map = {
            "fièvre": "Depuis combien de temps avez-vous de la fièvre ?",
            "maux de tête": "Avez-vous une sensibilité à la lumière ou au bruit ?",
            "douleur thoracique": "La douleur est-elle aiguë ou diffuse ?",
        }
        return questions_map.get(symptom.lower(), "Pouvez-vous donner plus de détails sur vos symptômes ?")

class MedicalAIPlugin:
    """Plugin qui analyse la gravité des symptômes."""
    @kernel_function(description="Vérifie la gravité d'un symptôme médical.")
    def check_symptom_severity(self, symptom: Annotated[str, "Symptôme décrit par l'utilisateur"]) -> str:
        """Retourne une évaluation de la gravité du symptôme."""
        severity_map = {
            "fièvre": "Modérée",
            "maux de tête": "Léger",
            "douleur thoracique": "Sévère",
            "perte de connaissance": "Critique"
        }
        return severity_map.get(symptom.lower(), "Inconnu - consultez un médecin.")

class PharmacistPlugin:
    """Plugin qui recommande des médicaments en fonction du diagnostic."""
    @kernel_function(description="Recommande un médicament adapté à un symptôme.")
    def recommend_medication(self, symptom: Annotated[str, "Symptôme décrit par l'utilisateur"]) -> str:
        """Retourne une suggestion de médicament (avec précautions)."""
        medication_map = {
            "fièvre": "Paracétamol (500mg, toutes les 6h, max 3 jours)",
            "maux de tête": "Ibuprofène (200mg, toutes les 8h, avec précaution si problème gastrique)",
            "douleur thoracique": "Aucun médicament recommandé - Consultez un médecin immédiatement",
        }
        return medication_map.get(symptom.lower(), "Aucun médicament recommandé - Consultez un pharmacien.")
print("Classe DoctorPlugin définie")
print("Classes DoctorPlugin, MedicalAIPlugin, PharmacistPlugin definies")


Classe DoctorPlugin définie


Classes DoctorPlugin, MedicalAIPlugin, PharmacistPlugin definies


### Exercise 1: Allergy Verification Plugin

The current plugins cover follow-up questions, severity, and medications. A crucial plugin is missing: **allergy verification**. The objective is to create `AllergyPlugin`, which allows the pharmacist agent to verify whether a recommended medication is compatible with the patient's declared allergies.

**Objective**: implement a class `AllergyPlugin` with a `check_allergy` method annotated with `@kernel_function` that returns a warning if the medication is incompatible with a known allergy.

**Hints**:
- # Step 1: Define a dictionary mapping allergies to contraindicated medications
- # Step 2: Implement the `check_allergy` method with `Annotated` type annotations
- # Hint: use the pattern of the existing plugins (DoctorPlugin, PharmacistPlugin)

In [1]:
class AllergyPlugin:
    """Plugin de verification des allergies medicamenteuses."""
    # TODO etudiant : implementer le plugin
    
    @kernel_function(description="Verifie si un medicament est compatible avec les allergies du patient.")
    def check_allergy(
        self, 
        medication: Annotated[str, "Nom du medicament"], 
        allergy: Annotated[str, "Allergie declaree du patient"]
    ) -> str:
        # Etape 1 : definir les contre-indications connues
        contraindications = {}  # TODO etudiant : dictionnaire allergie -> medicaments
        
        # Etape 2 : verifier et retourner le resultat
        result = None  # TODO etudiant : logique de verification
        return result  # TODO etudiant : retourner le message approprie

print("Exercice a completer : AllergyPlugin")

Exercice a completer


## Kernel Creation

In [8]:
# Création du kernel
kernel = create_kernel()
print("Kernel instancie")


Kernel instancie


## Adding plugins for each agent

In [9]:
# Ajout des plugins pour chaque agent
kernel.add_plugin(DoctorPlugin(), plugin_name="doctor")
kernel.add_plugin(MedicalAIPlugin(), plugin_name="medical")
kernel.add_plugin(PharmacistPlugin(), plugin_name="pharmacist")
print("Kernel configuré avec le plugin médical")
print("Classes plugins medicaux definies")
print("Kernel configure avec le plugin medical")


Kernel configuré avec le plugin médical
Classes plugins medicaux definies
Kernel configure avec le plugin medical


## System prompts: defining the behavior of each agent

System prompts are the instructions that guide the behavior of each agent. They establish the role, constraints, and expected response style. Good prompt design is essential to avoid undesirable behaviors (premature diagnosis, unfounded recommendation).

In [10]:
DOCTOR_PROMPT = """
Vous êtes un médecin généraliste. Vous posez d'abord des questions pour mieux comprendre les symptômes de l'utilisateur,
puis vous donnez un diagnostic probable basé sur votre expertise médicale. 
Ne donnez jamais de diagnostic sans avoir recueilli assez d'informations.
"""

AI_MEDICAL_PROMPT = """
Vous êtes une IA médicale spécialisée en diagnostic. Analysez les symptômes fournis et proposez un diagnostic basé sur des statistiques et des études médicales. 
Soyez clair et donnez plusieurs hypothèses si nécessaire.
"""

PHARMACIST_PROMPT = """
Vous êtes un pharmacien qualifié. En fonction du diagnostic fourni, vous recommandez les médicaments appropriés. 
Mentionnez toujours les précautions d'utilisation et la nécessité d'une consultation médicale avant la prise de médicaments.
"""
print("Prompt médical configuré")
print("Prompts medicaux configures")


Prompt médical configuré
Prompts medicaux configures


### Exercise 2: Adapt the prompts for a pediatric scenario

The current prompts are configured for an adult. The objective is to adapt the system for a **pediatric consultation**: the doctor must use language suited to children, the medical AI must take pediatric specifics into account (different dosage, specific vital signs), and the pharmacist must mention precautions for children.

**Objective**: write the three system prompts adapted to the pediatric context.

**Hints**:
- # Step 1: Modify the doctor's prompt for language accessible to children
- # Step 2: Adapt the medical AI prompt for pediatric dosages and signs
- # Hint: add constraints such as "always mention the recommended weight for dosage"

In [1]:
# Exercice 2 : Prompts adaptes au contexte pediatric
# TODO etudiant : rediger les trois prompts pediatric

PEDIATRIC_DOCTOR_PROMPT = ""    # Etape 1 : langage enfantin, questions adaptees
PEDIATRIC_AI_PROMPT = ""        # Etape 2 : dosages enfant, signes specific
PEDIATRIC_PHARMACIST_PROMPT = ""  # TODO etudiant : precautions enfant, posologie poids

print("Exercice a completer : prompts pediatric")

Exercice a completer


## Creating Agents

The three system prompts define the behavior and constraints of each agent. The doctor must ask questions before diagnosing, the medical AI analyzes symptoms using a statistical approach, and the pharmacist recommends treatments while reminding users of the precautions for use. This separation of roles is fundamental in a multi-agent architecture.

In [11]:
# Création des agents
doctor_agent = ChatCompletionAgent(
    kernel=kernel,
    name="Docteur_Humain",
    instructions=DOCTOR_PROMPT,
)

ai_medical_agent = ChatCompletionAgent(
    kernel=kernel,
    name="IA_Medicale",
    instructions=AI_MEDICAL_PROMPT,
)
print("Agents de conversation créés")
print("Kernel configure avec plugin medical")
print("Agents de conversation crees")


Agents de conversation créés
Kernel configure avec plugin medical
Agents de conversation crees


## Configuration so that the medical agent automatically calls the plugins

In [12]:
settings = kernel.get_prompt_execution_settings_from_service_id("openai")
settings.function_choice_behavior = FunctionChoiceBehavior.Auto()
ai_medical_agent.arguments = KernelArguments(settings=settings)

pharmacist_agent = ChatCompletionAgent(
    kernel=kernel,
    name="Pharmacien",
    instructions=PHARMACIST_PROMPT,
)
print("Paramètres d'exécution configurés")
print("Parametres d execution configures")


Paramètres d'exécution configurés
Parametres d execution configures


## Definition of a termination strategy

The `FunctionChoiceBehavior.Auto()` setting allows the `IA_Medicale` agent to automatically invoke the plugins registered in the kernel (symptom severity, recommendations). Without this configuration, the agent would only have its textual instructions to formulate its responses.

In [13]:
class MedicalTerminationStrategy(TerminationStrategy):
    async def should_terminate(self, agent, history):
        return len(history) >= 6  # On limite à 6 échanges
print("Stratégie de terminaison médicale définie")
print("Strategie de terminaison medicale definie")


Stratégie de terminaison médicale définie
Strategie de terminaison medicale definie


### Exercise 3: Termination strategy based on diagnosis

The current strategy stops after 6 messages. The objective is to implement a smarter strategy that detects when a **probable diagnosis** has been formulated by the medical AI (presence of words such as "diagnostic", "hypothese", "probablement" in the last message).

**Objective**: create `DiagnosticTerminationStrategy` that stops the conversation as soon as a diagnosis is issued or after a maximum of 8 exchanges.

**Hints**:
- # Step 1: Define a list of keywords indicative of a diagnosis
- # Step 2: Check for their presence in the content of the last message
- # Hint: combine the diagnosis condition with a maximum exchange counter

In [1]:
class DiagnosticTerminationStrategy(TerminationStrategy):
    # TODO etudiant : implementer la detection de diagnostic
    DIAGNOSTIC_KEYWORDS = []  # Etape 1 : mots-cles de diagnostic
    MAX_EXCHANGES = 8
    
    async def should_terminate(self, agent, history):
        # Etape 2 : verifier diagnostic OU limite d'echanges
        result = False  # TODO etudiant : remplacer par la logique
        return result

print("Exercice a completer : DiagnosticTerminationStrategy")

Exercice a completer


## Creating the group chat with a termination strategy

The `MedicalTerminationStrategy` class inherits from `TerminationStrategy` and implements `should_terminate`. Here, the condition is simple: after 6 messages in the history, the dialogue stops. This approach avoids infinite loops while allowing enough exchanges for a complete diagnosis.

In [14]:
chat = AgentGroupChat(
    agents=[doctor_agent, ai_medical_agent, pharmacist_agent],
    termination_strategy=MedicalTerminationStrategy()  # Ajout de la stratégie
)
print("Chat de groupe médical configuré")
print("Prompts medicaux configures")
print("Chat de groupe medical configure")


Chat de groupe médical configuré
Prompts medicaux configures
Chat de groupe medical configure


## Function to run the dialogue

The `run_medical_chat` function orchestrates the complete dialogue: it collects the initial symptoms via `input()`, passes the message to the `AgentGroupChat`, then iterates over the agents' responses. The cycle continues until the termination strategy declares the consultation complete.

In interactive mode (`BATCH_MODE = False`), the user can respond to each agent. In batch mode, the call is captured by the `try/except` block below.

In [15]:
async def run_medical_chat():
    logger.info("🚀 Début de la consultation médicale IA")
    chat_history = ChatHistory()
    
    symptoms = input("Décrivez vos symptômes : ")
    chat_history.add_user_message(symptoms)
    
    while True:
        async for message in chat.invoke():
            logger.info(f"[{message.role}] {message.name}: {message.content}")
            print(f"{message.name}: {message.content}")
            
            if message.name not in ["Pharmacien"]:
                user_response = input("👉 Votre réponse : ")
                chat_history.add_user_message(user_response)
        
        if chat.is_complete:
            break
    
    logger.info("🏥 Consultation terminée.")
print("Fonction de chat médical prête")
print("Fonction run_medical_chat prete")


Fonction de chat médical prête
Fonction run_medical_chat prete


In [16]:
try:
    await run_medical_chat()
except (EOFError, KeyboardInterrupt, Exception) as e:
    print(f"[Batch] Consultation interactive ignoree ({type(e).__name__}: {str(e)[:80]})")

2026-05-15 07:15:50,367 [INFO] 🚀 Début de la consultation médicale IA


[Batch] Consultation interactive ignoree (StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.)


## What we built

This use case illustrates several advanced Semantic Kernel concepts:

| Concept | Implementation |
|---------|----------------|
| **Specialized agents** | Three distinct roles (Doctor, Medical AI, Pharmacist) with dedicated system instructions |
| **Plugins `@kernel_function`** | Each agent has a plugin with specific functions (follow-up questions, severity assessment, medication recommendations) |
| **`AgentGroupChat`** | Multi-agent orchestration with automatic handoff between the three participants |
| **Termination strategy** | `MedicalTerminationStrategy` stops the dialogue after 6 exchanges to avoid infinite loops |
| **`FunctionChoiceBehavior.Auto()`** | The Medical AI agent can automatically call kernel plugins to enrich its responses |

### Key points to remember

1. **Separation of responsibilities**: each agent has a specific role, its own instructions, and its own tools. This modularity makes it easier to debug and evolve the system.

2. **Plugins as tools**: `@kernel_function` exposes capabilities that agents can invoke via `FunctionChoiceBehavior.Auto()`. The LLM decides when and which plugin to call based on the context.

3. **Control strategies**: the `TerminationStrategy` is essential in an `AgentGroupChat` to prevent agents from conversing indefinitely. Other strategies exist (agent selection, message filtering).

### Going further

- Add a simulated **Patient** agent that describes symptoms realistically
- Implement a persistent consultation history (storage in a database)
- Use a vector model to search for similar medical cases in a knowledge base
- Add ethical guardrails (refusal to diagnose certain conditions, systematic referral to a healthcare professional)